In [2]:
import pandas as pd
import numpy as np
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb
import os

print("All imports successful!")

All imports successful!


In [3]:
df = pd.read_csv('../data/processed/filtered_complaints.csv')

print("Shape:", df.shape)
print("\nProduct distribution:")
print(df['product_category'].value_counts())

Shape: (2092, 9)

Product distribution:
product_category
Savings Account    896
Credit Card        874
Money Transfer     200
Personal Loan      122
Name: count, dtype: int64


In [4]:
# Since we only have 2092 rows, use all of them
# but document that we're using a stratified approach

print("Using all available complaints with narrative:")
print(df['product_category'].value_counts())
print(f"\nTotal: {len(df)} complaints")

# This IS our stratified sample - all products represented proportionally
df_sample = df.copy()
print("\nSample ready!")

Using all available complaints with narrative:
product_category
Savings Account    896
Credit Card        874
Money Transfer     200
Personal Loan      122
Name: count, dtype: int64

Total: 2092 complaints

Sample ready!


In [5]:
# Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,        # each chunk is max 500 characters
    chunk_overlap=50,      # 50 character overlap between chunks
    length_function=len,   # measure length in characters
)

# Create chunks with metadata
all_chunks = []

for _, row in df_sample.iterrows():
    # Split the narrative into chunks
    chunks = text_splitter.split_text(row['cleaned_narrative'])
    
    # For each chunk, store the text AND metadata
    for i, chunk in enumerate(chunks):
        all_chunks.append({
            'text': chunk,
            'complaint_id': str(row['complaint_id']),
            'product_category': row['product_category'],
            'product': row['product'],
            'issue': row['issue'],
            'company': row['company'],
            'state': str(row['state']),
            'date_received': str(row['date_received']),
            'chunk_index': i,
            'total_chunks': len(chunks)
        })

chunks_df = pd.DataFrame(all_chunks)

print(f"Total complaints: {len(df_sample)}")
print(f"Total chunks created: {len(chunks_df)}")
print(f"Average chunks per complaint: {len(chunks_df)/len(df_sample):.1f}")
print(f"\nSample chunk:")
print(chunks_df.iloc[0]['text'])

Total complaints: 2092
Total chunks created: 6347
Average chunks per complaint: 3.0

Sample chunk:
a card was opened under my name by a fraudster. i received a notice from that an account was just opened under my name. i reached out to to state that this activity was unauthorized and not me. confirmed this was fraudulent and immediately closed the card. however, they have failed to remove this from the three credit agencies and this fraud is now impacting my credit score based on a hard credit pull done by that was done by a fraudster.


In [6]:
print("Loading embedding model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded!")

# Test it on one sentence so you can see what an embedding looks like
test_embedding = model.encode("I was charged twice for the same transaction")
print(f"\nEmbedding shape: {test_embedding.shape}")
print(f"First 10 numbers: {test_embedding[:10]}")

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\acer\Downloads\10 Acadamy\projects\Week_7\rag-complaint-chatbot\venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\acer\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded!

Embedding shape: (384,)
First 10 numbers: [ 0.01944029  0.0161465  -0.00618338  0.03902833  0.00206876 -0.08623882
  0.04450645 -0.05378115  0.07705721 -0.02943368]


In [7]:
print("Embedding all chunks... this will take a few minutes")

# Extract just the text from all chunks
texts = chunks_df['text'].tolist()

# Embed all at once in batches (batch_size=32 means 32 chunks at a time)
embeddings = model.encode(
    texts, 
    batch_size=32,
    show_progress_bar=True  # shows a progress bar
)

print(f"\nDone!")
print(f"Embeddings shape: {embeddings.shape}")
# Should be (6347, 384) — 6347 chunks, each with 384 numbers

Embedding all chunks... this will take a few minutes


Batches:   0%|          | 0/199 [00:00<?, ?it/s]


Done!
Embeddings shape: (6347, 384)


In [8]:
# Initialize ChromaDB - saves to disk
client = chromadb.PersistentClient(path='../vector_store/chroma_db')

# Create a collection (like a table in a database)
# if it already exists, delete and recreate
try:
    client.delete_collection("complaints")
except:
    pass

collection = client.create_collection(
    name="complaints",
    metadata={"hnsw:space": "cosine"}  # use cosine similarity for search
)

print("ChromaDB collection created!")
print("Now adding chunks... this may take a moment")

# Add in batches of 500
batch_size = 500

for i in range(0, len(chunks_df), batch_size):
    batch = chunks_df.iloc[i:i+batch_size]
    batch_embeddings = embeddings[i:i+batch_size]
    
    collection.add(
        ids=[f"chunk_{j}" for j in range(i, i+len(batch))],
        embeddings=batch_embeddings.tolist(),
        documents=batch['text'].tolist(),
        metadatas=batch[['complaint_id','product_category','product',
                         'issue','company','state',
                         'date_received','chunk_index']].to_dict('records')
    )
    print(f"Added chunks {i} to {i+len(batch)}")

print(f"\nDone! Total chunks in vector store: {collection.count()}")

ChromaDB collection created!
Now adding chunks... this may take a moment
Added chunks 0 to 500
Added chunks 500 to 1000
Added chunks 1000 to 1500
Added chunks 1500 to 2000
Added chunks 2000 to 2500
Added chunks 2500 to 3000
Added chunks 3000 to 3500
Added chunks 3500 to 4000
Added chunks 4000 to 4500
Added chunks 4500 to 5000
Added chunks 5000 to 5500
Added chunks 5500 to 6000
Added chunks 6000 to 6347

Done! Total chunks in vector store: 6347


In [9]:
# Test query
query = "problems with credit card billing"

# Embed the query using the same model
query_embedding = model.encode(query).tolist()

# Search for top 3 most similar chunks
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

print(f"Query: '{query}'")
print(f"\nTop 3 most relevant chunks:\n")

for i, (doc, metadata) in enumerate(zip(results['documents'][0], results['metadatas'][0])):
    print(f"Result {i+1}:")
    print(f"  Product: {metadata['product_category']}")
    print(f"  Issue: {metadata['issue']}")
    print(f"  Text: {doc[:200]}...")
    print()

Query: 'problems with credit card billing'

Top 3 most relevant chunks:

Result 1:
  Product: Credit Card
  Issue: Struggling to pay your bill
  Text: and they need to write their wrongs. i still have to pay these credit cards my accounts should not have been closed as they were when i called on . i was not even told i was in danger of account closu...

Result 2:
  Product: Credit Card
  Issue: Other features, terms, or problems
  Text: us bank told me they would add a 50.00 promotional credit to my rewards balance for the credit card becuase they couldnt remove an incorrect charge. so i had to pay the incorrect charge and they told ...

Result 3:
  Product: Credit Card
  Issue: Fees or interest
  Text: charges being applied to the card are a late fee and interest charges. i was advised during my initial call to , speaking to representative that because these charges have been applied to the card, ha...



In [10]:
# Cell 9 - Save chunks to processed folder for reference
chunks_df.to_csv('../data/processed/chunks.csv', index=False)
print(f"Saved {len(chunks_df)} chunks to chunks.csv")
print(f"Vector store saved at: ../vector_store/chroma_db")

Saved 6347 chunks to chunks.csv
Vector store saved at: ../vector_store/chroma_db
